# Comp Lab 2 — Designing a Novel Protein with ESM3

**BIO 462/594 | Thu Nov 5, 2026 | OSC 40 GB GPU | Teams of 2**

In Lab 1 you compared a model's *opinion* against curated evidence. Today there is no ground truth at all —
you will use **ESM3**, a generative protein language model trained on billions of years of evolution, to
**design a protein that has never existed**, then decide how much of the model's own confidence you actually buy.

**How this works:** ESM3 is a masked language model, like the ones that autocomplete your sentences.
You give it a *prompt* where some positions are filled in and others are masked (`_` means
"model, you fill this in"), and it iteratively unmasks the rest. You can fix an active-site motif and ask
for a new scaffold, fix a pattern of cysteines, or mask everything and design from nothing.

**Your team's job:** pick your design problem from the menu (Part 2), build the prompt, generate 3
candidates, evaluate them with the model's own confidence, and pick your best design to pitch in the
share-out.

---
## Part 0 — Setup (run this, then wait for your instructor)

The model weights are pre-staged on OSC project storage — this cell points the HuggingFace client at
them and loads ESM3 onto the GPU. **The first load takes 1–2 minutes.** Run it now, while your
instructor frames the session.

In [1]:
import os
# Load weights OFFLINE from project storage, with no student token or download.
os.environ['HF_HOME'] = os.environ.get('ESM3_HF_CACHE', '/fs/ess/PNS0503/hf_cache')
os.environ['HF_HUB_CACHE'] = os.path.join(os.environ['HF_HOME'], 'hub')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Lab 2 requires a GPU. Restart Classroom Jupyter with 1 GPU and select the BIO 462/594 Lab 2 - ESM3 kernel.')
device = 'cuda'
print('device:', torch.cuda.get_device_name(0))

from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein, GenerationConfig

model = ESM3.from_pretrained('esm3-open').to(device)
print('ESM3-open loaded (1.4B parameters)')

device: NVIDIA A100-PCIE-40GB MIG 7g.40gb
ESM3-open loaded (1.4B parameters)


---
## Part 1 — Your design brief

Every team gets a design problem from the menu. Set your team number and read your brief — the
*story* is what you pitch in the share-out; the *scaffold* is what you actually feed the model.

In [ ]:
import pandas as pd
menu = pd.read_csv('data/design_menu.csv')

TEAM = 1   # <-- YOUR TEAM NUMBER (1-10)

row = menu[menu.team == TEAM].iloc[0]
print(f"TEAM {TEAM}: {row['problem']}")
print()
print('Story:', row['story'])
print()
print('Expected outcome:', row['expected'])
print()
print('INTENSITY:', row['intensity'], '/ 5 -', row['intensity_why'])
print()
print('Generation settings: sequence steps =', row['seq_steps'],
      '| temperature =', row['seq_temp'], '| structure steps =', row['struct_steps'])

**Think, then discuss (2 min):** what does it mean that a single model can take *either* a fixed
motif, a pattern of amino acids, or nothing at all — and return a plausible protein? Write one sentence
your team would use to explain ESM3 to a doctor.

---
## Part 2 — Build your prompt

The prompt is an `ESMProtein` whose `sequence` mixes **fixed** residues and **masked** positions (`_`).
Run the cell for your scaffold type below.

In [ ]:
scaffold_type = row['scaffold_type']

if scaffold_type == 'masked_sequence':
    # Your scaffold is a string: fixed residues + '_' masks
    prompt = ESMProtein(sequence=row['scaffold'])
    n_masked = row['scaffold'].count('_')
    print(f'prompt length {len(row.scaffold)}, {n_masked} masked positions for ESM3 to design')

elif scaffold_type == 'motif_pdb':
    # Your scaffold fixes a REAL motif (sequence + 3D coordinates) from PDB 1ITU.
    # ESM3 must invent a new scaffold around it — in sequence AND in structure.
    import numpy as np
    motif = np.load(row['scaffold'])
    motif_seq = str(motif['sequence'])
    motif_coords = motif['atom37']
    L = 150
    full_seq = ['_'] * L
    full_seq[30:30 + len(motif_seq)] = list(motif_seq)   # place the motif inside a masked frame
    full_coords = np.full((L, 37, 3), np.nan, dtype=np.float32)  # NaN = "you decide the 3D shape"
    full_coords[30:30 + len(motif_seq)] = motif_coords
    prompt = ESMProtein(sequence=''.join(full_seq), coordinates=full_coords)
    print(f'motif "{motif_seq}" fixed at residues 31-{30 + len(motif_seq)} of a {L}-aa frame')

elif scaffold_type == 'rescue':
    # You start from a real p53 fragment (residues 157-246) carrying the Y220C
    # cancer mutation. Part 3 folds it, then round-trips:
    # structure -> new sequence -> new structure.
    prompt = ESMProtein(sequence=row['scaffold'])
    print(f'starting sequence ({len(row.scaffold)} aa, p53 residues 157-246 with the Y220C cancer mutation):')
    print(row['scaffold'])

print()
print('prompt ready:', scaffold_type)

**Your turn 1.** Change ONE thing about your prompt — shift the motif position, add a fixed glycine,
extend a mask — and predict out loud what will change. You will test that prediction in Part 3.

---
## Part 3 — Generate!

ESM3 generates in two passes: first the **sequence** track (iteratively unmasking amino acids), then the
**structure** track (predicting the 3D fold of whatever sequence it just wrote). Each pass is a loop of
forward passes — `num_steps` is how many. **Temperature** trades safety for diversity: 0.7 is
conservative, 1.0 is adventurous.

Generate **3 candidates** so you have something to choose between in Part 5.

In [ ]:
N_CANDIDATES = 3   # <-- dry-run classes may set this to 1

candidates = []
if scaffold_type == 'rescue':
    # The design cycle in miniature:
    # step 1 - fold the bad starting sequence as-is
    folded = model.generate(prompt, GenerationConfig(track='structure', num_steps=int(row['struct_steps'])))
    folded.to_pdb(f'team{TEAM}_candidate1_original.pdb')
    candidates.append({'i': 1, 'sequence': prompt.sequence, 'pdb': f'team{TEAM}_candidate1_original.pdb'})
    print('candidate 1: the original (bad) sequence, folded')
    # step 2 - inverse folding: KEEP the structure, ask ESM3 to write a new sequence for it
    folded.sequence = None
    seq_gen = model.generate(folded, GenerationConfig(track='sequence', num_steps=int(row['seq_steps']),
                                                      temperature=float(row['seq_temp'])))
    # step 3 - fold the new sequence
    seq_gen.coordinates = None
    struct_gen = model.generate(seq_gen, GenerationConfig(track='structure', num_steps=int(row['struct_steps'])))
    struct_gen.to_pdb(f'team{TEAM}_candidate2_roundtrip.pdb')
    candidates.append({'i': 2, 'sequence': seq_gen.sequence, 'pdb': f'team{TEAM}_candidate2_roundtrip.pdb'})
    print('candidate 2: the round-tripped redesign')
else:
    for i in range(N_CANDIDATES):
        seq_gen = model.generate(
            prompt,
            GenerationConfig(track='sequence', num_steps=int(row['seq_steps']),
                             temperature=float(row['seq_temp'])),
        )
        struct_gen = model.generate(
            seq_gen,
            GenerationConfig(track='structure', num_steps=int(row['struct_steps'])),
        )
        pdb_file = f'team{TEAM}_candidate{i + 1}.pdb'
        struct_gen.to_pdb(pdb_file)
        candidates.append({'i': i + 1, 'sequence': seq_gen.sequence, 'pdb': pdb_file})
        print(f'candidate {i + 1}: {len(seq_gen.sequence)} aa -> {pdb_file}')

print('done')

**Your turn 2.** Compare your candidate sequences (rescue teams: original vs round-tripped). Where do candidates differ most — inside your fixed
positions, or in the masked regions? (If they differ inside your *fixed* positions, check your prompt.)

---
## Part 4 — Look at your designs

Render each candidate in 3D, colored by the model's per-residue confidence (pLDDT, stored in the
PDB B-factor column — the same convention AlphaFold DB uses).

In [ ]:
import py3Dmol

def show(pdb_file):
    view = py3Dmol.view(width=450, height=380)
    view.addModel(open(pdb_file).read(), 'pdb')
    view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 50, 'max': 90}}})
    view.zoomTo()
    return view

show(candidates[0]['pdb'])

In [ ]:
# per-residue pLDDT from the B-factor column (same convention as AlphaFold DB)
import matplotlib.pyplot as plt

def plddt_from_pdb(pdb_file):
    vals = []
    for line in open(pdb_file):
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            vals.append(float(line[60:66]))
    return vals

plt.figure(figsize=(7, 3.2))
for c in candidates:
    plt.plot(plddt_from_pdb(c['pdb']), label=f"candidate {c['i']}", alpha=0.8)
plt.axhline(50, color='grey', ls=':', lw=1)
plt.xlabel('residue')
plt.ylabel('pLDDT')
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(f'team{TEAM}_plddt.png', dpi=140)
plt.show()

---
## Part 5 — Pick your best design

Score every candidate on the model's confidence, then choose. There is no right answer here - what
matters is the evidence you use to defend your choice, which is exactly what the report asks for.

In [ ]:
import numpy as np
summary = []
for c in candidates:
    p = plddt_from_pdb(c['pdb'])
    summary.append({'candidate': c['i'],
                    'mean pLDDT': round(float(np.mean(p)), 1),
                    '% residues < 50': round(100 * float(np.mean(np.array(p) < 50)), 0),
                    'length': len(p)})
pd.DataFrame(summary)

**Your turn 3 (the share-out slide).** Fill in your team's answers:
1. Our design problem was: ...
2. We fixed this in the prompt: ... and let ESM3 design this: ...
3. Our best candidate is candidate N because: ...
4. The confidence pattern we saw (where high, where low) and our hypothesis why: ...
5. One thing we would change in the next design cycle: ...

---
## Part 6 — Wrap-up

**Think, then discuss (3 min):**
1. Lab 1 asked whether you trust a model's score on proteins that already exist. Today you trusted a
   model to invent one. What is the difference between trusting a prediction and trusting an invention?
2. ESM3 was trained on natural proteins. Your design has no natural history. Which parts of your
   confidence score do you expect to be *most* and *least* reliable, and why?
3. If a lab actually synthesized your best candidate and it did not fold — whose fault is it: the
   prompt, the model, or the confidence score? (There is no agreed answer in the field. Defend one.)

**The Design Report** (30% of the lab portfolio) is assigned today and due **Thursday, November 12**.
Your notebook, your three PDB files, and your Part 5 answers are your source material.

---
## BIO 594 extension

Modify code and document the change (syllabus requirement). Suggested: sweep temperature
(0.5 / 0.7 / 1.0) at fixed prompt and steps, plot mean pLDDT and sequence diversity across
candidates, and report where the diversity–quality tradeoff sits for your problem. Or implement the
full round-trip (structure → inverse-fold → structure) on your own design and quantify the change.